In [ ]:
import os
at_colab = "COLAB_GPU" in os.environ

if at_colab:
    %pip install pyrosm

In [ ]:
import pandas as pd
import pyrosm
from pathlib import Path
# Web
import requests
from urllib.parse import urlparse


# Notebook specific
from tqdm.notebook import tqdm
# global definitions
_data_path = Path( './data/' )

In [ ]:
def download_with_progress(
    url: str,
    file_path: Path,
    chunk_size: int = 1024 * 1024
) -> None:
    '''
    Downloads a file from a given URL with a progress bar.

    Args:
        url: The URL of the file to download.
        file_path: The local file path to save the downloaded file.
        chunk_size: The size of each chunk in bytes. Defaults to 1MB.

    Returns:
        None
    '''
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))  # Total file size in bytes
    num_chunks = total_size // chunk_size if total_size else None  # Number of chunks if known

    with file_path.open('wb') as local_file, tqdm(
        total=num_chunks, unit='chunk', disable=(num_chunks is None)
    ) as progress_bar:
        for chunk in response.iter_content(chunk_size=chunk_size):
            local_file.write(chunk)
            progress_bar.update(1 if num_chunks else len(chunk))  # Update based on chunk count or bytes


In [ ]:
def download_file_if_needed(
    url: str,
    chunk_size: int = 1024 * 1024
) -> str:
    '''
    Downloads a file from the given URL if it does not already exist.

    Args:
        url: The URL of the file to download.
        chunk_size: The size of each chunk in bytes. Defaults to 1MB.

    Returns:
        The local file path as a string.
    '''
    file_name = Path(urlparse(url).path).name
    path = _data_path.joinpath(file_name)

    if not (path.exists() and path.is_file()):
        path.parent.mkdir(parents=True, exist_ok=True)
        download_with_progress(url, path, chunk_size)

    return str(path)

In [ ]:
gpkd = download_file_if_needed('https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_NLD.gpkg')

In [ ]:
import geopandas as gpd
gpd.read_file(gpkd,layer="ADM_ADM_2").explore()

In [ ]:
boundaries_name = 'nl_boundaries.pkl'

In [ ]:
osm = pyrosm.OSM(pyrosm.get_data('netherlands'))

In [ ]:
if Path(boundaries_name).is_file():
    boundaries = pd.read_pickle(boundaries_name)
else:
    boundaries = osm.get_boundaries()
    boundaries.to_pickle(boundaries_name)

In [ ]:
municipalities = boundaries[(boundaries.admin_level=='8') & (boundaries['addr:country'].isna())].dropna(axis=1, how='all')

In [ ]:
municipalities.columns

In [ ]:
municipalities.explore()

In [ ]:
import ast
expanded_tags = pd.json_normalize(municipalities['tags'].apply(ast.literal_eval))

In [ ]:
expanded_tags['population'].dropna().astype(int).sum()

In [ ]:
municipalities = municipalities.loc[~(expanded_tags['population'].isna().values)]

In [ ]:
municipalities.explore(tooltip='name')

In [ ]:
municipalities[municipalities.name.str.contains('Capelle')].explore(
    tooltip=False,
    style_kwds={'color': 'green', 'weight': 5, 'fill': False}
)
